# Discord Voice Bot - Google Colab

Run your Discord voice bot on Google Colab with free GPU (T4).

**Steps:**
1. Set Runtime -> Change runtime type -> **T4 GPU**
2. Fill in your tokens below
3. Run all cells

In [ ]:
#@title 1. Settings (fill in your tokens)
DISCORD_BOT_TOKEN = '' #@param {type:"string"}
OPENAI_API_KEY = '' #@param {type:"string"}
OPENAI_BASE_URL = '' #@param {type:"string"}

#@markdown ---
#@markdown ### LLM Settings
LLM_MODEL = 'gemini-2.5-flash-lite' #@param {type:"string"}
LLM_MAX_TOKENS = 150 #@param {type:"integer"}
LLM_TEMPERATURE = 0.9 #@param {type:"number"}

#@markdown ---
#@markdown ### STT Settings
STT_BACKEND = 'realtime' #@param ['realtime', 'onnx']
STT_MODEL = 'large-v3' #@param ['large-v3', 'medium', 'small', 'base', 'tiny']
STT_LANGUAGE = 'ru' #@param {type:"string"}

#@markdown ---
#@markdown ### TTS Settings
TTS_ENGINE = 'edge' #@param {type:"string"}
TTS_VOICE = 'ru-RU-DmitryNeural' #@param {type:"string"}

#@markdown ---
#@markdown ### Bot Personality
BOT_NAME = 'Alex' #@param {type:"string"}

In [ ]:
#@title 2. Install system dependencies
!apt-get update -qq
!apt-get install -y -qq ffmpeg mpv libmpv-dev > /dev/null 2>&1
print('ffmpeg and mpv installed')

In [ ]:
#@title 3. Clone repo and install Python dependencies
!git clone https://github.com/karkqm/discord-voice-bott.git bot 2>/dev/null || (cd bot && git pull)
%cd bot
!git checkout feature/audio-streaming-refactor

!pip install -q py-cord[voice]==2.6.1 openai python-dotenv PyNaCl Pillow aiohttp numpy
!pip install -q RealtimeSTT RealtimeTTS edge-tts
!pip install -q faster-whisper
print('Dependencies installed')

In [ ]:
#@title 4. Check GPU
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f'GPU: {gpu_name}')
    print(f'CUDA: {torch.version.cuda}')
else:
    print('No GPU found! Go to Runtime -> Change runtime type -> T4 GPU')

In [ ]:
#@title 5. Create .env
env_content = f"""DISCORD_BOT_TOKEN={DISCORD_BOT_TOKEN}
OPENAI_API_KEY={OPENAI_API_KEY}
OPENAI_BASE_URL={OPENAI_BASE_URL}

LLM_MODEL={LLM_MODEL}
LLM_MAX_TOKENS={LLM_MAX_TOKENS}
LLM_TEMPERATURE={LLM_TEMPERATURE}
IS_LOCAL_LLM=false

STT_BACKEND={STT_BACKEND}
STT_MODEL={STT_MODEL}
STT_LANGUAGE={STT_LANGUAGE}
TTS_ENGINE={TTS_ENGINE}
TTS_VOICE={TTS_VOICE}

SCREEN_CAPTURE_INTERVAL=5
SCREEN_CAPTURE_ENABLED=false

BOT_NAME={BOT_NAME}
BOT_ALIASES=бот,алекс,андрей,слышь
BARGE_IN_SENSITIVITY=0.5
"""

with open('.env', 'w') as f:
    f.write(env_content)
print('.env created')
print(f'STT: {STT_BACKEND} / {STT_MODEL}')
print(f'TTS: {TTS_ENGINE} / {TTS_VOICE}')

In [ ]:
#@title 6. Run bot
!python bot.py